# 02 – Train LeFusion

Run on the **Windows GPU machine**. Three steps:
1. Imports
2. Load configs (`configs/data/data.yaml`, `configs/train/lefusion_train.yaml`) and override values if needed
3. Train

The data (split + patches) is prepared automatically; patches already cut by `01_dataset.ipynb` are reused.
Progress, validation loss and preview samples are printed while training. Everything of a run goes to `outputs/<run_name>/`: `train_log.txt`, `metrics.csv`, `config.yaml`, `samples/*.png`, `checkpoints/` (`latest.pt`, `best.pt`, `step_*.pt`).
Training is seeded (`seed` in `configs/train/lefusion_train.yaml`), so the same settings give the same run.
To stop: *Interrupt kernel* – `latest.pt` is saved and running cell 3 again resumes from it. A new `run_name` starts a fresh run; an existing run is never overwritten.
Same from a terminal: `python scripts/run_train.py` (see `--help`).

In [ ]:
# 1) Imports
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (_here, *_here.parents) if (p / "src" / "aneurysm").is_dir())  # project folder
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from aneurysm.config import load_data_config
from aneurysm.config import load_train_config
from aneurysm.training import train
from aneurysm.utils.env import describe_environment

describe_environment()

In [ ]:
# 2) Configs: values come from the YAML files; put changes for this run in the dicts below
DATA_OVERRIDES = {
    # "patch_depth": 24,          # smaller patch if memory is still too high
}
TRAIN_OVERRIDES = {
    "run_name": "pilot_64x64x32_small",   # new name -> fresh run folder in outputs/
    # smaller network for the 4 GB Quadro T2000
    "base_channels": 32,          # channels at level 1 (paper: 64)
    "dim_mults": [1, 2, 4, 4],    # channels per level: 32, 64, 128, 128 (paper: 64, 128, 256, 512)
    "attn_heads": 4,              # attention heads (paper: 8)
    "num_workers": 0,             # safest on Windows + Jupyter
}

data_cfg = load_data_config(PROJECT_ROOT / "configs" / "data" / "data.yaml", project_root=PROJECT_ROOT,
                            overrides=DATA_OVERRIDES)
train_cfg = load_train_config(PROJECT_ROOT / "configs" / "train" / "lefusion_train.yaml", project_root=PROJECT_ROOT,
                              overrides=TRAIN_OVERRIDES)

print("Patches:", data_cfg.patch_tag)
print("Run    :", train_cfg.run_name, "| channels", [train_cfg.base_channels * m for m in train_cfg.dim_mults],
      "| heads", train_cfg.attn_heads, "| batch", train_cfg.batch_size, "x accum", train_cfg.grad_accum,
      "| steps", train_cfg.max_steps)

In [ ]:
# 3) Train
result = train(data_cfg, train_cfg)